# LoRA vs Full Fine-Tuning on RoBERTa-base

This notebook clones the project repo, installs dependencies, and runs all four
experiments:

| Mode | Dataset |
|---|---|
| baseline | SST-2 |
| lora | SST-2 |
| baseline | MNLI |
| lora | MNLI |

Results are collected and printed as a comparison table at the end.

## 1. Setup

In [ ]:
!git clone https://github.com/da-luce/cs5782_final_project.git
%cd cs5782_final_project

In [ ]:
!pip install -r requirements.txt -q

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

## 2. SST-2 Experiments

In [ ]:
!python code/train.py --mode baseline --dataset sst2

In [ ]:
!python code/train.py --mode lora --dataset sst2

## 3. MNLI Experiments

In [ ]:
!python code/train.py --mode baseline --dataset mnli

In [ ]:
!python code/train.py --mode lora --dataset mnli

## 4. Results Comparison

In [ ]:
import json
import os

experiments = [
    ("baseline", "sst2"),
    ("lora",     "sst2"),
    ("baseline", "mnli"),
    ("lora",     "mnli"),
]

rows = []
for mode, dataset in experiments:
    path = f"results/{mode}_{dataset}.json"
    if not os.path.exists(path):
        print(f"Missing results for {mode}/{dataset} — skipping")
        continue
    with open(path) as f:
        info = json.load(f)
    accuracy = info["eval_results"].get("eval_accuracy", float("nan"))
    trainable = info["trainable_params"]["trainable"]
    trainable_pct = info["trainable_params"]["trainable_pct"]
    total = info["trainable_params"]["total"]
    time_min = info["training_time_sec"] / 60
    rows.append((mode, dataset, accuracy, trainable, trainable_pct, total, time_min))

header = f"{'Mode':<12} {'Dataset':<8} {'Accuracy':>10} {'Trainable params':>18} {'Trainable %':>12} {'Total params':>14} {'Train time (min)':>18}"
sep = "-" * len(header)
print(header)
print(sep)
for mode, dataset, acc, tr, tr_pct, tot, t in rows:
    print(f"{mode:<12} {dataset:<8} {acc:>10.4f} {tr:>18,} {tr_pct:>11.4f}% {tot:>14,} {t:>17.1f}m")